# Planejamento Ótimo de Rotina de Treinos

Este notebook implementa, os três modelos desenvolvidos para o planejamento ótimo de rotinas de musculação, usando Gurobi como solver:

1. **Bloco 1 — Arquitetura dos Parâmetros:** Definição da estrutura para os parâmetros de entrada do modelo
2. **Bloco 2 — Catálogo de Exercícios e Instanciação:** Definição do catálogo de exercícios (ativação muscular $\alpha_{j,m}$, teto $\hat{I}_j$ e demanda $\gamma_j$) e inicialização da instância base.
3. **Bloco 3 — Construtores dos Três Modelos:**
   * **Modelo 1 — MILP Ingênuo:** Maximiza o estímulo linear bruto, apenas com fadiga sistêmica e inter-dias.
   * **Modelo 2 — MINLP Bilinear:** Incorpora fadiga muscular local ($FadLoc$), penalidade terminal e o ciclo de produção--consumo com interrupção de recuperação ($Y_{i,m}, P_{i,m}, G_{i,m}$).
   * **Modelo 3 — MILP Linearizado:** Equivalente exato ao Modelo 2, utilizando linearização via **Big-M** para eliminar os produtos bilineares e retornar à classe MILP.
4. **Bloco 4 — Resolução e Resultados:** Resolução da instância, comparação direta do valor objetivo máximo entre os dois últimos modelos, tabela de alocação de exercícios com intensidades, matriz de fadiga muscular ao final de cada dia e resultado consolidado.
5. **Bloco 5 — Análise de Sensibilidade Paramétrica:** Varredura rápida no modelo linearizado final sobre a taxa de recuperação ($r$), tolerância sinergista ($\epsilon$) e pesos terminais ($\rho, \lambda$).

## Bloco 1: Arquitetura dos Parâmetros

Centralizamos todos os conjuntos, índices e parâmetros do problema na dataclass `ParametrosModelo`. O limitante superior diário $M_i = 1 - (1 - r)^{i-1}$ é calculado automaticamente a partir da série geométrica de $P_{i,m}$ para uso na linearização Big-M do Modelo 3.

In [11]:
import copy
import time
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

print(f"Versão do Gurobi: {gp.gurobi.version()}")


@dataclass
class ParametrosModelo:
    D: int = 7
    K_max: int = 6
    M: List[str] = field(default_factory=lambda: [
        "Peito", "Costas", "Quadríceps", "Posterior", "Ombros", "Bíceps", "Tríceps"
    ])
    Disp: Dict[int, int] = field(default_factory=lambda: {
        1: 1, 2: 1, 3: 1, 4: 0, 5: 1, 6: 1, 7: 1
    })
    pref: Dict[str, float] = field(default_factory=lambda: {
        "Peito": 1.15,
        "Costas": 1.15,
        "Quadríceps": 1.10,
        "Posterior": 1.05,
        "Ombros": 1.00,
        "Bíceps": 0.90,
        "Tríceps": 0.90,
    })
    dados_exercicios: Dict[str, dict] = field(default_factory=dict)

    # Microparâmetros fisiológicos e de recuperação
    c: float = 0.50          # Fator de piso de intensidade
    beta: float = 0.25       # Sensibilidade à fadiga sistêmica intra-treino
    beta_loc: float = 0.35   # Sensibilidade à fadiga muscular local intra-treino
    a: float = 0.55          # Coeficiente global de geração aguda de fadiga inter-dias
    b: float = 0.45          # Coeficiente global de retenção residual de fadiga inter-dias
    eps: float = 0.22        # Tolerância de ativação sinergista
    r: float = 0.55          # Taxa diária de decaimento/conversão do estímulo (delta = r)
    rho: float = 0.35        # Peso de valoração residual do reservatório terminal P[D+1, m]
    lam: float = 0.25        # Peso de penalização da fadiga terminal Fad[D+1, m]

    @property
    def I(self) -> List[int]:
        return list(range(1, self.D + 1))

    @property
    def I_plus(self) -> List[int]:
        return list(range(1, self.D + 2))

    @property
    def K(self) -> List[int]:
        return list(range(1, self.K_max + 1))

    @property
    def K_plus(self) -> List[int]:
        return list(range(1, self.K_max + 2))

    @property
    def J(self) -> List[str]:
        return list(self.dados_exercicios.keys())

    @property
    def I_hat(self) -> Dict[str, float]:
        return {j: self.dados_exercicios[j]["I_hat"] for j in self.J}

    @property
    def gamma(self) -> Dict[str, float]:
        return {j: self.dados_exercicios[j]["gamma"] for j in self.J}

    @property
    def alpha(self) -> Dict[Tuple[str, str], float]:
        return {
            (j, m): self.dados_exercicios[j]["alpha"].get(m, 0.0)
            for j in self.J for m in self.M
        }

    @property
    def M_big(self) -> Dict[int, float]:
        # Limitante Big-M diário exato para r * P[i, m]
        return {i: 1.0 - (1.0 - self.r) ** (i - 1) for i in self.I}

Versão do Gurobi: (14, 0, 0)


## Bloco 2: Instanciação dos Exercícios e Microparâmetros

Definimos uma instância de porte médio com 15 exercícios distribuídos entre 7 grupos musculares, um horizonte de 7 dias (6 dias disponíveis e 1 dia de descanso total) e até 5 slots por sessão. Garantimos que a matriz de ativação seja normalizada ($\sum_{m \in M} \alpha_{j,m} = 1$).

In [12]:
CATALOGO_EXERCICIOS = {
    "Supino_Reto_Barra": {
        "I_hat": 0.85, "gamma": 0.24,
        "alpha": {"Peito": 0.65, "Tríceps": 0.20, "Ombros": 0.15}
    },
    "Supino_Inclinado_Halter": {
        "I_hat": 0.80, "gamma": 0.19,
        "alpha": {"Peito": 0.60, "Ombros": 0.25, "Tríceps": 0.15}
    },
    "Crucifixo_Cabo": {
        "I_hat": 0.75, "gamma": 0.10,
        "alpha": {"Peito": 0.85, "Ombros": 0.15}
    },
    "Remada_Curvada_Barra": {
        "I_hat": 0.85, "gamma": 0.25,
        "alpha": {"Costas": 0.70, "Bíceps": 0.20, "Ombros": 0.10}
    },
    "Puxada_Frontal": {
        "I_hat": 0.80, "gamma": 0.17,
        "alpha": {"Costas": 0.75, "Bíceps": 0.25}
    },
    "Remada_Baixa_Cabo": {
        "I_hat": 0.78, "gamma": 0.15,
        "alpha": {"Costas": 0.80, "Bíceps": 0.20}
    },
    "Agachamento_Livre": {
        "I_hat": 0.88, "gamma": 0.32,
        "alpha": {"Quadríceps": 0.70, "Posterior": 0.30}
    },
    "Leg_Press": {
        "I_hat": 0.85, "gamma": 0.23,
        "alpha": {"Quadríceps": 0.80, "Posterior": 0.20}
    },
    "Cadeira_Extensora": {
        "I_hat": 0.75, "gamma": 0.10,
        "alpha": {"Quadríceps": 1.00}
    },
    "Stiff_Barra": {
        "I_hat": 0.85, "gamma": 0.29,
        "alpha": {"Posterior": 0.75, "Costas": 0.25}
    },
    "Mesa_Flexora": {
        "I_hat": 0.75, "gamma": 0.10,
        "alpha": {"Posterior": 1.00}
    },
    "Desenvolvimento_Militar": {
        "I_hat": 0.82, "gamma": 0.21,
        "alpha": {"Ombros": 0.70, "Tríceps": 0.30}
    },
    "Elevacao_Lateral": {
        "I_hat": 0.75, "gamma": 0.08,
        "alpha": {"Ombros": 1.00}
    },
    "Rosca_Direta_Barra": {
        "I_hat": 0.75, "gamma": 0.09,
        "alpha": {"Bíceps": 1.00}
    },
    "Triceps_Polia": {
        "I_hat": 0.75, "gamma": 0.09,
        "alpha": {"Tríceps": 1.00}
    },
}

params_base = ParametrosModelo(dados_exercicios=CATALOGO_EXERCICIOS)

# Verificação da normalização das ativações
for j in params_base.J:
    soma = sum(params_base.alpha[j, m] for m in params_base.M)
    assert np.isclose(soma, 1.0), f"Ativação de {j} soma {soma} != 1.0"

df_catalogo = pd.DataFrame([
    {
        "Exercicio": j,
        "I_hat": params_base.I_hat[j],
        "gamma": params_base.gamma[j],
        "Ativacoes": ", ".join(f"{m}: {int(params_base.alpha[j, m]*100)}%" for m in params_base.M if params_base.alpha[j, m] > 0)
    }
    for j in params_base.J
])
print(f"Instância carregada: {params_base.D} dias, {params_base.K_max} slots/dia, {len(params_base.J)} exercícios, {len(params_base.M)} músculos.")
display(df_catalogo)

Instância carregada: 7 dias, 6 slots/dia, 15 exercícios, 7 músculos.


,Exercicio,I_hat,gamma,Ativacoes
0,Supino_Reto_Barra,0.85,0.24,"Peito: 65%, Ombros: 15%, Tríceps: 20%"
1,Supino_Inclinado_Halter,0.80,0.19,"Peito: 60%, Ombros: 25%, Tríceps: 15%"
2,Crucifixo_Cabo,0.75,0.10,"Peito: 85%, Ombros: 15%"
3,Remada_Curvada_Barra,0.85,0.25,"Costas: 70%, Ombros: 10%, Bíceps: 20%"
4,Puxada_Frontal,0.80,0.17,"Costas: 75%, Bíceps: 25%"
5,Remada_Baixa_Cabo,0.78,0.15,"Costas: 80%, Bíceps: 20%"
6,Agachamento_Livre,0.88,0.32,"Quadríceps: 70%, Posterior: 30%"
7,Leg_Press,0.85,0.23,"Quadríceps: 80%, Posterior: 20%"
8,Cadeira_Extensora,0.75,0.10,Quadríceps: 100%
9,Stiff_Barra,0.85,0.29,"Costas: 25%, Posterior: 75%"


## Bloco 3: Construtores dos Três Modelos

Construímos três funções independentes para deixar clara a evolução da modelagem:

1. **`construir_milp_ingenuo(params)`** — Formulação base original (maximização linear do estímulo bruto, sem fadiga local e sem ciclo de recuperação).
2. **`construir_minlp_bilinear(params)`** — Formulação fisiológica completa com fadiga muscular local ($FadLoc$), penalidade terminal e o ciclo de produção--consumo ($P_{i,m}, G_{i,m}, Y_{i,m}$). Como envolve produtos entre variáveis binárias e contínuas ($Ex \cdot FadTr$, $Ex \cdot FadLoc$ e $Y \cdot P$), é um problema **MIQCP não-convexo** (`NonConvex = 2`).
3. **`construir_milp_linearizado(params)`** — Diferencia-se do Modelo 2 exclusivamente por substituir os termos bilineares por restrições lineares equivalentes usando **Big-M** ($\beta \gamma_j$, $\beta_{loc} \alpha_{j,m}$ e $M_i = 1 - (1-r)^{i-1}$), devolvendo o problema à classe **MILP puro** sem perda de exatidão.

In [13]:
# ==============================================================================
# MODELO 1: MILP INGÊNUO (BASE)
# ==============================================================================
def construir_milp_ingenuo(params: ParametrosModelo, silencioso: bool = True):
    model = gp.Model("1_MILP_Ingenuo")
    if silencioso:
        model.Params.OutputFlag = 0

    I, J, K, M = params.I, params.J, params.K, params.M
    I_hat, gamma, alpha = params.I_hat, params.gamma, params.alpha

    Ex = model.addVars(I, J, K, vtype=GRB.BINARY, name="Ex")
    Int = model.addVars(
        [(i, j, k) for i in I for j in J for k in K],
        lb=0.0, ub={(i, j, k): I_hat[j] for i in I for j in J for k in K},
        vtype=GRB.CONTINUOUS, name="Int"
    )
    Fad = model.addVars(params.I_plus, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="Fad")
    FadTr = model.addVars(I, K, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadTr")

    # [FO] Maximiza o estímulo bruto linear ponderado pela prioridade (pref) e ativação (alpha)
    model.setObjective(
        gp.quicksum(params.pref[m] * alpha[j, m] * Int[i, j, k] for i in I for j in J for k in K for m in M),
        GRB.MAXIMIZE
    )

    for i in I:
        for k in K:
            # [R1 - Disponibilidade de Slot] Aloca no máximo 1 exercício por slot apenas em dias disponíveis (Disp[i] = 1)
            model.addConstr(gp.quicksum(Ex[i, j, k] for j in J) <= params.Disp[i])
        for k in range(1, params.K_max):
            # [R2 - Ordenação Sequencial] Preenche os slots em ordem contínua, evitando "buracos" vazios na sessão
            model.addConstr(gp.quicksum(Ex[i, j, k + 1] for j in J) <= gp.quicksum(Ex[i, j, k] for j in J))
            # [R3 - Acúmulo de Fadiga Sistêmica] Atualiza a fadiga intra-treino somando a demanda energética do slot anterior (para k < K_max)
            model.addConstr(FadTr[i, k + 1] == FadTr[i, k] + gp.quicksum(gamma[j] * Int[i, j, k] for j in J))
        # [R4 - Condição Inicial de Fadiga Sistêmica] Inicia toda sessão diária com fadiga sistêmica zerada
        model.addConstr(FadTr[i, 1] == 0.0)

        for j in J:
            # [R5 - Unicidade Diária] Proíbe repetir o mesmo exercício mais de uma vez no mesmo dia
            model.addConstr(gp.quicksum(Ex[i, j, k] for k in K) <= 1)
            for k in K:
                # [R6 - Piso de Intensidade] Exige intensidade mínima viável (c * I_hat) caso o exercício seja selecionado
                model.addConstr(Int[i, j, k] >= params.c * I_hat[j] * Ex[i, j, k])
                # [R7 - Teto de Intensidade] Limita a carga ao teto físico I_hat e força Int = 0 se o exercício não for alocado
                model.addConstr(Int[i, j, k] <= I_hat[j] * Ex[i, j, k])
                # [R8 - Degradação Sistêmica (Ingênua)] Reduz o teto de intensidade conforme FadTr acumulada; relaxada quando Ex = 0
                model.addConstr(
                    Int[i, j, k] <= I_hat[j] - params.beta * gamma[j] * FadTr[i, k] + (params.beta * gamma[j]) * (1 - Ex[i, j, k])
                )

        for m in M:
            # [R9 - Capacidade Muscular Diária] Limita o volume diário absorvido pelo músculo à sua capacidade residual (1 - Fad)
            model.addConstr(gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J for k in K) <= 1.0 - Fad[i, m])

    for m in M:
        # [R10 - Condição Inicial de Fadiga Crônica] Inicia o horizonte de planejamento com os músculos descansados
        model.addConstr(Fad[1, m] == 0.0)
        for i in I:
            # [R11 - Transição de Fadiga Inter-Dias] Acumula fadiga aguda gerada no dia (a) e retém fração residual do dia anterior (b)
            model.addConstr(
                Fad[i + 1, m] == params.a * gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J for k in K) + params.b * Fad[i, m]
            )

    model.update()
    return model, {"Ex": Ex, "Int": Int, "Fad": Fad, "FadTr": FadTr}


# ==============================================================================
# MODELO 2: MINLP BILINEAR NÃO-CONVEXO (CONCEITUAL COMPLETO)
# ==============================================================================
def construir_minlp_bilinear(params: ParametrosModelo, silencioso: bool = True):
    model = gp.Model("2_MINLP_Bilinear")
    if silencioso:
        model.Params.OutputFlag = 0
    model.Params.NonConvex = 2

    I, I_plus = params.I, params.I_plus
    K, K_plus = params.K, params.K_plus
    J, M = params.J, params.M
    I_hat, gamma, alpha = params.I_hat, params.gamma, params.alpha

    Ex = model.addVars(I, J, K, vtype=GRB.BINARY, name="Ex")
    Y = model.addVars(I, M, vtype=GRB.BINARY, name="Y")
    Int = model.addVars(
        [(i, j, k) for i in I for j in J for k in K],
        lb=0.0, ub={(i, j, k): I_hat[j] for i in I for j in J for k in K},
        vtype=GRB.CONTINUOUS, name="Int"
    )
    V = model.addVars(I, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="V")
    Fad = model.addVars(I_plus, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="Fad")
    FadTr = model.addVars(I, K_plus, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadTr")
    FadLoc = model.addVars(I, K_plus, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadLoc")
    P = model.addVars(I_plus, M, lb=0.0, ub=1.0 / params.r, vtype=GRB.CONTINUOUS, name="P")
    G = model.addVars(I, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="G")

    # [FO] Maximiza a hipertrofia realizada (G), soma valor residual do reservatório terminal (rho*P) e penaliza fadiga final (lam*Fad)
    model.setObjective(
        gp.quicksum(params.pref[m] * G[i, m] for i in I for m in M)
        + params.rho * gp.quicksum(params.pref[m] * P[params.D + 1, m] for m in M)
        - params.lam * gp.quicksum(Fad[params.D + 1, m] for m in M),
        GRB.MAXIMIZE
    )

    for i in I:
        for k in K:
            # [R1 - Disponibilidade de Slot] Aloca no máximo 1 exercício por slot apenas nos dias disponíveis
            model.addConstr(gp.quicksum(Ex[i, j, k] for j in J) <= params.Disp[i])
        for k in range(1, params.K_max):
            # [R2 - Ordenação Sequencial] Garante preenchimento contínuo dos slots sem lacunas vazias
            model.addConstr(gp.quicksum(Ex[i, j, k + 1] for j in J) <= gp.quicksum(Ex[i, j, k] for j in J))

        for j in J:
            # [R3 - Unicidade Diária] Impede repetir o mesmo exercício no mesmo dia
            model.addConstr(gp.quicksum(Ex[i, j, k] for k in K) <= 1)
            for k in K:
                # [R4 - Piso de Intensidade] Impõe intensidade mínima eficaz (c * I_hat) quando o exercício é escolhido
                model.addConstr(Int[i, j, k] >= params.c * I_hat[j] * Ex[i, j, k])
                # [R5 - Teto de Intensidade] Limita a carga ao teto I_hat e zera Int quando Ex = 0
                model.addConstr(Int[i, j, k] <= I_hat[j] * Ex[i, j, k])
                # [R6 - Degradação Sistêmica Bilinear] Penaliza a intensidade pela fadiga sistêmica acumulada (produto Ex * FadTr)
                model.addConstr(Int[i, j, k] <= Ex[i, j, k] * (I_hat[j] - params.beta * gamma[j] * FadTr[i, k]))
                for m in M:
                    if alpha[j, m] > 0:
                        # [R7 - Degradação Muscular Local Bilinear] Penaliza a intensidade pela fadiga local de cada músculo ativado (produto Ex * FadLoc)
                        model.addConstr(Int[i, j, k] <= Ex[i, j, k] * (I_hat[j] - params.beta_loc * alpha[j, m] * FadLoc[i, k, m]))

        # [R8 - Condição Inicial de Fadiga Sistêmica] Inicia a sessão diária com fadiga sistêmica nula
        model.addConstr(FadTr[i, 1] == 0.0)
        for k in K:
            # [R9 - Acúmulo de Fadiga Sistêmica] Propaga FadTr até K_max + 1, garantindo que o último slot também respeite o teto unitário
            model.addConstr(FadTr[i, k + 1] == FadTr[i, k] + gp.quicksum(gamma[j] * Int[i, j, k] for j in J))

        for m in M:
            # [R10 - Definição do Volume Diário] Agrega o estímulo total recebido pelo músculo m no dia i
            model.addConstr(V[i, m] == gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J for k in K))
            # [R11 - Capacidade Muscular Diária] Restringe o volume diário à capacidade disponível (1 - Fad)
            model.addConstr(V[i, m] <= 1.0 - Fad[i, m])
            # [R12 - Ativação de Sessão com Tolerância Sinergista] Ativa Y = 1 se o volume exceder eps; volumes <= eps não interrompem a recuperação
            model.addConstr(V[i, m] <= params.eps + (1.0 - params.eps) * Y[i, m])
            # [R13 - Condição Inicial de Fadiga Local] Inicializa a fadiga local no slot 1 com a fadiga crônica do início do dia
            model.addConstr(FadLoc[i, 1, m] == Fad[i, m])
            for k in K:
                # [R14 - Acúmulo de Fadiga Local] Acumula a fadiga muscular slot a slot até K_max + 1, incentivando alternância na sessão
                model.addConstr(FadLoc[i, k + 1, m] == FadLoc[i, k, m] + gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J))

    for m in M:
        # [R15 - Condição Inicial de Fadiga Crônica] Inicia o horizonte sem fadiga crônica acumulada
        model.addConstr(Fad[1, m] == 0.0)
        # [R16 - Condição Inicial do Reservatório de Estímulo] Inicia o horizonte sem estímulo pendente de conversão
        model.addConstr(P[1, m] == 0.0)
        for i in I:
            # [R17 - Transição de Fadiga Inter-Dias] Evolui a fadiga crônica até D + 1 a partir do volume diário (a) e retenção residual (b)
            model.addConstr(Fad[i + 1, m] == params.a * V[i, m] + params.b * Fad[i, m])
            # [R18 - Dinâmica do Reservatório de Estímulo] Decai o reservatório à taxa r (com delta = r) e soma o novo volume diário V
            model.addConstr(P[i + 1, m] == (1.0 - params.r) * P[i, m] + V[i, m])
            # [R19 - Conversão de Estímulo Bilinear] Converte fração r de P em crescimento G apenas em dias de descanso muscular (Y = 0)
            model.addConstr(G[i, m] == (1 - Y[i, m]) * params.r * P[i, m])

    model.update()
    return model, {"Ex": Ex, "Y": Y, "Int": Int, "V": V, "Fad": Fad, "FadTr": FadTr, "FadLoc": FadLoc, "P": P, "G": G}


# ==============================================================================
# MODELO 3: MILP LINEARIZADO EXATO (BIG-M)
# ==============================================================================
def construir_milp_linearizado(params: ParametrosModelo, silencioso: bool = True):
    model = gp.Model("3_MILP_Linearizado")
    if silencioso:
        model.Params.OutputFlag = 0

    I, I_plus = params.I, params.I_plus
    K, K_plus = params.K, params.K_plus
    J, M = params.J, params.M
    I_hat, gamma, alpha = params.I_hat, params.gamma, params.alpha
    M_big = params.M_big

    Ex = model.addVars(I, J, K, vtype=GRB.BINARY, name="Ex")
    Y = model.addVars(I, M, vtype=GRB.BINARY, name="Y")
    Int = model.addVars(
        [(i, j, k) for i in I for j in J for k in K],
        lb=0.0, ub={(i, j, k): I_hat[j] for i in I for j in J for k in K},
        vtype=GRB.CONTINUOUS, name="Int"
    )
    V = model.addVars(I, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="V")
    Fad = model.addVars(I_plus, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="Fad")
    FadTr = model.addVars(I, K_plus, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadTr")
    FadLoc = model.addVars(I, K_plus, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="FadLoc")
    P = model.addVars(I_plus, M, lb=0.0, ub=1.0 / params.r, vtype=GRB.CONTINUOUS, name="P")
    G = model.addVars(I, M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="G")

    # [FO] Mesma função objetivo do Modelo 2: maximiza hipertrofia realizada (G) + estímulo terminal (rho*P) - fadiga terminal (lam*Fad)
    model.setObjective(
        gp.quicksum(params.pref[m] * G[i, m] for i in I for m in M)
        + params.rho * gp.quicksum(params.pref[m] * P[params.D + 1, m] for m in M)
        - params.lam * gp.quicksum(Fad[params.D + 1, m] for m in M),
        GRB.MAXIMIZE
    )

    for i in I:
        for k in K:
            # [R1 - Disponibilidade de Slot] Aloca no máximo 1 exercício por slot apenas nos dias disponíveis
            model.addConstr(gp.quicksum(Ex[i, j, k] for j in J) <= params.Disp[i])
        for k in range(1, params.K_max):
            # [R2 - Ordenação Sequencial] Garante preenchimento contínuo dos slots sem lacunas vazias
            model.addConstr(gp.quicksum(Ex[i, j, k + 1] for j in J) <= gp.quicksum(Ex[i, j, k] for j in J))

        for j in J:
            # [R3 - Unicidade Diária] Impede repetir o mesmo exercício no mesmo dia
            model.addConstr(gp.quicksum(Ex[i, j, k] for k in K) <= 1)
            for k in K:
                # [R4 - Piso de Intensidade] Impõe intensidade mínima eficaz (c * I_hat) quando o exercício é escolhido
                model.addConstr(Int[i, j, k] >= params.c * I_hat[j] * Ex[i, j, k])
                # [R5 - Teto de Intensidade] Limita a carga ao teto I_hat e zera Int quando Ex = 0
                model.addConstr(Int[i, j, k] <= I_hat[j] * Ex[i, j, k])
                # [R6 - Degradação Sistêmica Linearizada (Big-M)] Substitui o produto Ex * FadTr usando Big-M apertado (beta * gamma)
                model.addConstr(
                    Int[i, j, k] <= I_hat[j] * Ex[i, j, k] - params.beta * gamma[j] * FadTr[i, k] + (params.beta * gamma[j]) * (1 - Ex[i, j, k])
                )
                for m in M:
                    if alpha[j, m] > 0:
                        # [R7 - Degradação Local Linearizada (Big-M)] Substitui o produto Ex * FadLoc usando Big-M apertado (beta_loc * alpha)
                        model.addConstr(
                            Int[i, j, k] <= I_hat[j] * Ex[i, j, k] - params.beta_loc * alpha[j, m] * FadLoc[i, k, m] + (params.beta_loc * alpha[j, m]) * (1 - Ex[i, j, k])
                        )

        # [R8 - Condição Inicial de Fadiga Sistêmica] Inicia a sessão diária com fadiga sistêmica nula
        model.addConstr(FadTr[i, 1] == 0.0)
        for k in K:
            # [R9 - Acúmulo de Fadiga Sistêmica] Propaga FadTr até K_max + 1, cobrindo também o último slot da sessão
            model.addConstr(FadTr[i, k + 1] == FadTr[i, k] + gp.quicksum(gamma[j] * Int[i, j, k] for j in J))

        for m in M:
            # [R10 - Definição do Volume Diário] Agrega o estímulo total recebido pelo músculo m no dia i
            model.addConstr(V[i, m] == gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J for k in K))
            # [R11 - Capacidade Muscular Diária] Restringe o volume diário à capacidade disponível (1 - Fad)
            model.addConstr(V[i, m] <= 1.0 - Fad[i, m])
            # [R12 - Ativação de Sessão com Tolerância Sinergista] Ativa Y = 1 se o volume exceder eps; volumes <= eps não interrompem a recuperação
            model.addConstr(V[i, m] <= params.eps + (1.0 - params.eps) * Y[i, m])
            # [R13 - Condição Inicial de Fadiga Local] Inicializa a fadiga local no slot 1 com a fadiga crônica do início do dia
            model.addConstr(FadLoc[i, 1, m] == Fad[i, m])
            for k in K:
                # [R14 - Acúmulo de Fadiga Local] Acumula a fadiga muscular slot a slot até K_max + 1
                model.addConstr(FadLoc[i, k + 1, m] == FadLoc[i, k, m] + gp.quicksum(alpha[j, m] * Int[i, j, k] for j in J))

    for m in M:
        # [R15 - Condição Inicial de Fadiga Crônica] Inicia o horizonte sem fadiga crônica acumulada
        model.addConstr(Fad[1, m] == 0.0)
        # [R16 - Condição Inicial do Reservatório de Estímulo] Inicia o horizonte sem estímulo pendente de conversão
        model.addConstr(P[1, m] == 0.0)
        for i in I:
            # [R17 - Transição de Fadiga Inter-Dias] Evolui a fadiga crônica até D + 1 a partir do volume diário (a) e retenção residual (b)
            model.addConstr(Fad[i + 1, m] == params.a * V[i, m] + params.b * Fad[i, m])
            # [R18 - Dinâmica do Reservatório de Estímulo] Decai o reservatório à taxa r (com delta = r) e soma o novo volume diário V
            model.addConstr(P[i + 1, m] == (1.0 - params.r) * P[i, m] + V[i, m])
            # [R19a - Linearização do Crescimento (Limite do Reservatório)] Limita o crescimento diário à parcela conversível r * P[i, m]
            model.addConstr(G[i, m] <= params.r * P[i, m])
            # [R19b - Linearização do Crescimento (Bloqueio Big-M)] Zera G se o músculo treinar (Y = 1) e relaxa via teto diário M_i se descansar (Y = 0)
            model.addConstr(G[i, m] <= M_big[i] * (1 - Y[i, m]))

    model.update()
    return model, {"Ex": Ex, "Y": Y, "Int": Int, "V": V, "Fad": Fad, "FadTr": FadTr, "FadLoc": FadLoc, "P": P, "G": G}

## Bloco 4: Resolução da Instância e Visualização dos Resultados

Resolvemos a instância para os três modelos e apresentamos os resultados de forma direta:
1. **Comparação de Valor Objetivo Máximo** entre o **Modelo 2 (MINLP Bilinear)** e o **Modelo 3 (MILP Linearizado)**, confirmando que a linearização via Big-M preserva o mesmo ótimo com ganho computacional.
2. **Alocação de Exercícios e Intensidades** ($Ex_{i,j,k} = 1$ e $Int_{i,j,k}$) por dia e por slot.
3. **Fadiga Muscular ao Final de Cada Dia** ($Fad_{i+1, m}$ para $i = 1, \dots, D$) para cada grupo muscular.
4. **Resumo do Resultado Final** (valor objetivo e hipertrofia realizada acumulada).

In [14]:
def extrair_tabelas_resultado(vars_dict, params: ParametrosModelo):
    # 1. Alocação de exercícios e intensidade por dia/slot
    linhas_aloc = []
    for i in params.I:
        if params.Disp[i] == 0:
            linhas_aloc.append({
                "Dia": i, "Slot": "-", "Exercicio": "DESCANSO",
                "Intensidade (Int)": 0.0, "Teto (I_hat)": 0.0, "% do Teto": "-"
            })
            continue
        for k in params.K:
            escolhido = [j for j in params.J if vars_dict["Ex"][i, j, k].X > 0.5]
            if escolhido:
                j = escolhido[0]
                val_int = vars_dict["Int"][i, j, k].X
                linhas_aloc.append({
                    "Dia": i,
                    "Slot": k,
                    "Exercicio": j,
                    "Intensidade (Int)": round(val_int, 3),
                    "Teto (I_hat)": params.I_hat[j],
                    "% do Teto": f"{100 * val_int / params.I_hat[j]:.1f}%",
                })
    df_aloc = pd.DataFrame(linhas_aloc)

    # 2. Fadiga acumulada ao final de cada dia i (equivalente a Fad[i+1, m])
    df_fad = pd.DataFrame(
        {m: [round(vars_dict["Fad"][i + 1, m].X, 3) for i in params.I] for m in params.M},
        index=[f"Fim do Dia {i}" for i in params.I]
    )
    return df_aloc, df_fad


# ------------------------------------------------------------------------------
# RESOLUÇÃO DOS MODELOS
# ------------------------------------------------------------------------------
# Modelo 1: MILP Ingênuo
m1, v1 = construir_milp_ingenuo(params_base, silencioso=False)
m1.Params.MIPGap = 1e-4
m1.Params.TimeLimit = 450.0
t0 = time.perf_counter()
m1.optimize()
t1 = time.perf_counter() - t0

# Modelo 2: MINLP Bilinear
m2, v2 = construir_minlp_bilinear(params_base, silencioso=False)
m2.Params.MIPGap = 1e-4
m2.Params.TimeLimit = 450.0
t0 = time.perf_counter()
m2.optimize()
t2 = time.perf_counter() - t0

# Modelo 3: MILP Linearizado Exato
m3, v3 = construir_milp_linearizado(params_base, silencioso=False)
m3.Params.MIPGap = 1e-4
m3.Params.TimeLimit = 450.0
t0 = time.perf_counter()
m3.optimize()
t3 = time.perf_counter() - t0
def montar_grade_alocacao(modelo, vars_dict, params: ParametrosModelo) -> pd.DataFrame:
    if modelo.SolCount == 0:
        return pd.DataFrame({"Status": ["Nenhuma solução viável encontrada"]})

    grade = {}
    for i in params.I:
        col = f"Dia {i}" + (" (Desc.)" if params.Disp[i] == 0 else "")
        coluna_slots = []
        for k in params.K:
            if params.Disp[i] == 0:
                coluna_slots.append("DESCANSO")
                continue
            escolhido = [j for j in params.J if vars_dict["Ex"][i, j, k].X > 0.5]
            if escolhido:
                j = escolhido[0]
                val_int = vars_dict["Int"][i, j, k].X
                coluna_slots.append(f"{j} ({val_int:.2f})")
            else:
                coluna_slots.append("-")
        grade[col] = coluna_slots

    return pd.DataFrame(grade, index=[f"Slot {k}" for k in params.K])


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

for titulo, m_atual, v_atual in [
    ("Modelo 1 — MILP Ingênuo", m1, v1),
    ("Modelo 2 — MINLP Bilinear", m2, v2),
    ("Modelo 3 — MILP Linearizado", m3, v3),
]:
    print(f"\n=== {titulo} ===")
    display(montar_grade_alocacao(m_atual, v_atual, params_base))

Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 450
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (win64 - Windows 11.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  450

Academic license 2885135 - for non-commercial use only - registered to gi___@poli.ufrj.br
Optimize a model with 2219 rows, 1358 columns and 9828 nonzeros (Max)
Model fingerprint: 0xc0db814c
Model has 630 linear objective coefficients
Variable types: 728 continuous, 630 integer (630 binary)
Coefficient statistics:
  Matrix range     [2e-02, 1e+00]
  Objective range  [9e-01, 1e+00]
  Bounds range     [8e-01, 1e+00]
  RHS range        [8e-01, 1e+00]

Found heuristic solution: objective -0.0000000
Presolve removed 433 rows and 213 columns
Presolve time: 0.03s
Presolved: 1786 rows, 1145 columns, 11448 nonzeros
Variable types:

,Dia 1,Dia 2,Dia 3,Dia 4 (Desc.),Dia 5,Dia 6,Dia 7
Slot 1,Desenvolvimento_Militar (0.82),Rosca_Direta_Barra (0.75),Remada_Baixa_Cabo (0.41),DESCANSO,Crucifixo_Cabo (0.75),Cadeira_Extensora (0.53),Supino_Reto_Barra (0.76)
Slot 2,Supino_Inclinado_Halter (0.79),Cadeira_Extensora (0.69),Cadeira_Extensora (0.48),DESCANSO,Desenvolvimento_Militar (0.82),Supino_Reto_Barra (0.84),Cadeira_Extensora (0.50)
Slot 3,Stiff_Barra (0.83),Remada_Curvada_Barra (0.82),Rosca_Direta_Barra (0.38),DESCANSO,Leg_Press (0.84),Stiff_Barra (0.63),Remada_Baixa_Cabo (0.63)
Slot 4,Remada_Curvada_Barra (0.81),Triceps_Polia (0.60),Supino_Reto_Barra (0.78),DESCANSO,Remada_Curvada_Barra (0.82),Remada_Curvada_Barra (0.45),Rosca_Direta_Barra (0.38)
Slot 5,Supino_Reto_Barra (0.80),Mesa_Flexora (0.53),Desenvolvimento_Militar (0.71),DESCANSO,Stiff_Barra (0.80),Triceps_Polia (0.61),Desenvolvimento_Militar (0.79)
Slot 6,Agachamento_Livre (0.80),Supino_Inclinado_Halter (0.75),Stiff_Barra (0.66),DESCANSO,Rosca_Direta_Barra (0.61),Rosca_Direta_Barra (0.38),Mesa_Flexora (0.50)



=== Modelo 2 — MINLP Bilinear ===


,Dia 1,Dia 2,Dia 3,Dia 4 (Desc.),Dia 5,Dia 6,Dia 7
Slot 1,Mesa_Flexora (0.65),Rosca_Direta_Barra (0.70),Cadeira_Extensora (0.38),DESCANSO,Puxada_Frontal (0.72),Stiff_Barra (0.80),-
Slot 2,Leg_Press (0.58),Desenvolvimento_Militar (0.66),Supino_Reto_Barra (0.80),DESCANSO,Elevacao_Lateral (0.40),Leg_Press (0.64),-
Slot 3,Triceps_Polia (0.74),Puxada_Frontal (0.74),Stiff_Barra (0.79),DESCANSO,Rosca_Direta_Barra (0.62),Agachamento_Livre (0.54),-
Slot 4,Supino_Inclinado_Halter (0.76),Elevacao_Lateral (0.37),Supino_Inclinado_Halter (0.40),DESCANSO,Triceps_Polia (0.71),Supino_Reto_Barra (0.81),-
Slot 5,Agachamento_Livre (0.77),Remada_Baixa_Cabo (0.56),Agachamento_Livre (0.54),DESCANSO,Remada_Curvada_Barra (0.42),Crucifixo_Cabo (0.43),-
Slot 6,Supino_Reto_Barra (0.75),-,-,DESCANSO,Desenvolvimento_Militar (0.56),-,-



=== Modelo 3 — MILP Linearizado ===


,Dia 1,Dia 2,Dia 3,Dia 4 (Desc.),Dia 5,Dia 6,Dia 7
Slot 1,Mesa_Flexora (0.69),Rosca_Direta_Barra (0.70),Cadeira_Extensora (0.38),DESCANSO,Remada_Baixa_Cabo (0.42),Stiff_Barra (0.80),-
Slot 2,Triceps_Polia (0.74),Puxada_Frontal (0.74),Stiff_Barra (0.79),DESCANSO,Rosca_Direta_Barra (0.64),Agachamento_Livre (0.54),-
Slot 3,Supino_Inclinado_Halter (0.76),Desenvolvimento_Militar (0.66),Agachamento_Livre (0.54),DESCANSO,Triceps_Polia (0.71),Leg_Press (0.64),-
Slot 4,Leg_Press (0.80),Remada_Baixa_Cabo (0.56),Supino_Reto_Barra (0.80),DESCANSO,Desenvolvimento_Militar (0.56),Supino_Reto_Barra (0.81),-
Slot 5,Supino_Reto_Barra (0.75),Elevacao_Lateral (0.38),Supino_Inclinado_Halter (0.40),DESCANSO,Elevacao_Lateral (0.45),Crucifixo_Cabo (0.43),-
Slot 6,Agachamento_Livre (0.51),-,-,DESCANSO,Puxada_Frontal (0.67),-,-


## Bloco 5: Análise de Sensibilidade Paramétrica (Modelo 3)

Por fim, avaliamos rapidamente como o **MILP Linearizado Final** responde à variação dos três principais parâmetros de calibragem:
1. **Taxa de conversão/recuperação ($r$):** Mede como a velocidade de recuperação altera o número de sessões musculares ativas ($\sum Y$) e o valor objetivo.
2. **Tolerância sinergista ($\epsilon$):** Verifica o impacto de flexibilizar o limite de volume secundário sobre o volume total e o número de ativações $Y$.
3. **Pesos terminais ($\rho, \lambda$):** Mostra como o volume do último dia disponível e a fadiga terminal respondem à penalização de fim de horizonte.

In [16]:
# ==============================================================================
# COMPARAÇÃO DE HIPERPARÂMETROS (MODELO 3 — MILP LINEARIZADO)
# ==============================================================================

# Caso 1: Recuperação Lenta (r = 0.25, frente à base r = 0.55)
p_caso1 = copy.deepcopy(params_base)
p_caso1.r = 0.25
p_caso1.rho = 0.15
m_c1, v_c1 = construir_milp_linearizado(p_caso1, silencioso=True)
m_c1.Params.MIPGap = 1e-4
m_c1.Params.TimeLimit = 300.0  # Comente esta linha para rodar sem limite de tempo
t0 = time.perf_counter()
m_c1.optimize()
t_c1 = time.perf_counter() - t0

# Caso 2: Tolerância Sinergista Rigorosa (eps = 0.05, frente à base eps = 0.22)
p_caso2 = copy.deepcopy(params_base)
p_caso2.eps = 0.05

m_c2, v_c2 = construir_milp_linearizado(p_caso2, silencioso=True)
m_c2.Params.MIPGap = 1e-4
m_c2.Params.TimeLimit = 300.0  # Comente esta linha para rodar sem limite de tempo
t0 = time.perf_counter()
m_c2.optimize()
t_c2 = time.perf_counter() - t0

# ------------------------------------------------------------------------------
# 1. TABELA COMPARATIVA GERAL
# ------------------------------------------------------------------------------
df_comp_hiper = pd.DataFrame([
    {
        "Caso": "Caso 1 — Recuperação Lenta (r = 0.25)",
        "Valor_Objetivo_Max": round(m_c1.ObjVal, 4) if m_c1.SolCount > 0 else None,
        "Tempo_s": round(t_c1, 3),
        "Gap_pct": round(m_c1.MIPGap * 100, 3) if m_c1.SolCount > 0 else None,
    },
    {
        "Caso": "Caso 2 — Tolerância Rigorosa (eps = 0.05)",
        "Valor_Objetivo_Max": round(m_c2.ObjVal, 4) if m_c2.SolCount > 0 else None,
        "Tempo_s": round(t_c2, 3),
        "Gap_pct": round(m_c2.MIPGap * 100, 3) if m_c2.SolCount > 0 else None,
    },
])
print("=== COMPARAÇÃO DOS CASOS DE HIPERPARÂMETROS (MODELO 3) ===")
display(df_comp_hiper)

# ------------------------------------------------------------------------------
# 2. RESULTADOS DETALHADOS DE CADA CASO (GRADE, FADIGA E RESUMO POR MÚSCULO)
# ------------------------------------------------------------------------------
for titulo, m_atual, v_atual, p_atual, tempo_atual in [
    ("CASO 1 — RECUPERAÇÃO LENTA (r = 0.25)", m_c1, v_c1, p_caso1, t_c1),
    ("CASO 2 — TOLERÂNCIA SINERGISTA RIGOROSA (eps = 0.05)", m_c2, v_c2, p_caso2, t_c2),
]:
    print(f"\n=== {titulo} | Objetivo Ótimo: {m_atual.ObjVal:.4f} | Tempo: {tempo_atual:.3f}s ===")
    _, df_fad_c = extrair_tabelas_resultado(v_atual, p_atual)

    print("\n--- Grade de Alocação e Intensidades ---")
    display(montar_grade_alocacao(m_atual, v_atual, p_atual))

    print("\n--- Fadiga Muscular ao Final de Cada Dia: Fad[i+1, m] ---")
    display(df_fad_c)

    df_resumo_c = pd.DataFrame({
        "Musculo": p_atual.M,
        "Prioridade (pref)": [p_atual.pref[m] for m in p_atual.M],
        "Dias_Treino_Ativo (Soma Y)": [sum(int(round(v_atual["Y"][i, m].X)) for i in p_atual.I) for m in p_atual.M],
        "Hipertrofia_Realizada (Soma G)": [round(sum(v_atual["G"][i, m].X for i in p_atual.I), 3) for m in p_atual.M],
        "Fadiga_Final (Fad[D+1])": [round(v_atual["Fad"][p_atual.D + 1, m].X, 3) for m in p_atual.M],
    })
    print("\n--- Resumo por Grupo Muscular ---")
    display(df_resumo_c)

=== COMPARAÇÃO DOS CASOS DE HIPERPARÂMETROS (MODELO 3) ===


,Caso,Valor_Objetivo_Max,Tempo_s,Gap_pct
0,Caso 1 — Recuperação Lenta (r = 0.25),9.2070,300.677,2.385
1,Caso 2 — Tolerância Rigorosa (eps = 0.05),13.1834,50.487,0.000



=== CASO 1 — RECUPERAÇÃO LENTA (r = 0.25) | Objetivo Ótimo: 9.2070 | Tempo: 300.677s ===

--- Grade de Alocação e Intensidades ---


,Dia 1,Dia 2,Dia 3,Dia 4 (Desc.),Dia 5,Dia 6,Dia 7
Slot 1,Cadeira_Extensora (0.75),Stiff_Barra (0.80),Agachamento_Livre (0.63),DESCANSO,Triceps_Polia (0.69),Crucifixo_Cabo (0.62),-
Slot 2,Supino_Inclinado_Halter (0.80),Mesa_Flexora (0.40),Remada_Baixa_Cabo (0.39),DESCANSO,Rosca_Direta_Barra (0.70),Cadeira_Extensora (0.38),-
Slot 3,Remada_Curvada_Barra (0.84),Rosca_Direta_Barra (0.45),Supino_Reto_Barra (0.80),DESCANSO,Mesa_Flexora (0.37),Remada_Baixa_Cabo (0.64),-
Slot 4,Rosca_Direta_Barra (0.69),Triceps_Polia (0.68),Supino_Inclinado_Halter (0.40),DESCANSO,Desenvolvimento_Militar (0.48),Agachamento_Livre (0.73),-
Slot 5,Supino_Reto_Barra (0.74),Desenvolvimento_Militar (0.58),Puxada_Frontal (0.44),DESCANSO,Stiff_Barra (0.62),Supino_Reto_Barra (0.56),-
Slot 6,Puxada_Frontal (0.55),Elevacao_Lateral (0.38),Cadeira_Extensora (0.38),DESCANSO,Elevacao_Lateral (0.50),Remada_Curvada_Barra (0.43),-



--- Fadiga Muscular ao Final de Cada Dia: Fad[i+1, m] ---


,Peito,Costas,Quadríceps,Posterior,Ombros,Bíceps,Tríceps
Fim do Dia 1,0.528,0.550,0.413,0.000,0.217,0.548,0.147
Fim do Dia 2,0.238,0.358,0.186,0.550,0.528,0.495,0.535
Fim do Dia 3,0.524,0.514,0.531,0.351,0.359,0.326,0.362
Fim do Dia 4,0.236,0.231,0.239,0.158,0.161,0.147,0.163
Fim do Dia 5,0.106,0.190,0.108,0.534,0.534,0.450,0.534
Fim do Dia 6,0.539,0.531,0.539,0.361,0.361,0.320,0.302
Fim do Dia 7,0.243,0.239,0.243,0.163,0.163,0.144,0.136


... (4 outputs left out)


--- Fadiga Muscular ao Final de Cada Dia: Fad[i+1, m] ---


,Peito,Costas,Quadríceps,Posterior,Ombros,Bíceps,Tríceps
Fim do Dia 1,0.000,0.550,0.550,0.550,0.000,0.550,0.000
Fim do Dia 2,0.550,0.248,0.248,0.248,0.550,0.248,0.550
Fim do Dia 3,0.247,0.525,0.525,0.525,0.275,0.525,0.247
Fim do Dia 4,0.111,0.236,0.236,0.236,0.124,0.236,0.111
Fim do Dia 5,0.539,0.106,0.106,0.106,0.538,0.106,0.539
Fim do Dia 6,0.242,0.539,0.539,0.539,0.269,0.539,0.242
Fim do Dia 7,0.109,0.243,0.243,0.243,0.121,0.243,0.109



--- Resumo por Grupo Muscular ---


,Musculo,Prioridade (pref),Dias_Treino_Ativo (Soma Y),Hipertrofia_Realizada (Soma G),Fadiga_Final (Fad[D+1])
0,Peito,1.15,3,1.579,0.109
1,Costas,1.15,3,1.851,0.243
2,Quadríceps,1.10,3,1.851,0.243
3,Posterior,1.05,3,1.851,0.243
4,Ombros,1.00,3,1.632,0.121
5,Bíceps,0.90,3,1.851,0.243
6,Tríceps,0.90,3,1.579,0.109
